# Un core · dosis de nutrición reducida

Sensibilidad de dosis al100%,75% y50% de los pulsos usados en LAB019–021 y LAB016–018, sin recalibración. theta_nv2 fijo, M original A=0.3,tau_r4h,tau_d12h, O₂ activo y transferencia constante del11 (liberación0.5870419885h⁻¹, escalaCsat2.5, ganancia0.77428737694). Las referencias son las del13; no se usa el ensayo descartado12.

La reducción se aplica **al salto de N en la única química y a la dosis que entra en M**. Por ello cambian potencialmente X,YAN,etanol,G,F y G+F, además de CO₂. No es otra sensibilidad de A. Temperaturas, iniciales, horarios, volumen usado en observaciones y todos los coeficientes quedan iguales. No se reduce la condición inicial de YAN ni el nutriente previo a inoculación.

## Procedencia de dosis

LAB004–009: hoja Eventos_operacion del Excel original registra protocolo Springferm Xtrem1.00g + FDA0.40g (p.ej.I3/I4,I8/I9,I13/I14 e I18/I22/I26). Las filas7–9 posteriores no documentan por sí solas masas efectivas de cada pulso; son registros de protocolo, no una comprobación gravimétrica. La química existente utiliza pulso_nut del conjunto normalizado; se imprime por LAB sin modificarlo.

LAB019–021: Mediciones-Offline-LAB019-021.xlsx, Nutricion!E5:H7: 80mg/L YAN,1g SFX,0.4g FDA y una antigua anotación de100mL. El usuario aclaró que se aplicó producto directamente y sin añadir solución; no se agrega volumen por nutrición.

LAB016–018: las masas no están verificadas en su Excel. El código histórico asumió1g SFX+0.4g FDA,20% de aporte YAN para ambos y2L, obteniendo140mg/L; LAB017 usa doble dosis en un solo pulso,280mg/L. Una auditoría antigua recoge una alternativa no confirmada de0.8g SFX+0.4g FDA (doble en LAB017). No se presenta ninguna de ellas como masa experimental confirmada.

Bajo el protocolo nominal1g+.4g en2L, las concentraciones de producto son0.5g/L SFX y0.2g/L FDA; el doble sería1 y0.4g/L. Estas cifras no validan la conversión a YAN. La equivalencia general es DeltaN=(fSFX*mSFX+fFDA*mFDA)/V: no se adopta el mismo20% para ambos como verdad experimental. Las reducciones de este ensayo escalan el YAN modelado sin decidir esa conversión pendiente.

## Política

Medianas horarias RAW convertidas por volumen remanente, sin filtrar predicciones. Mismos horizontes y muestras del11/13, LAB017 un pulso único. Roles previos de calibración y holdout no implican nuevos ajustes ni selección de dosis por error: son escenarios hipotéticos, no corrección de registros.

Una química por escenario y LAB (18 integraciones), ningún core0. La concentración de las sustancias se conserva al retirar una muestra bien mezclada; no se añade volumen de nutriente. Se mantienen las dosis nominales del lector, sin nueva renormalización por volumen. Transferencia histórica y Cd0=0. Figuras por pares con muestras: CO₂ completo/pico24h/cola desde32h y química E,G,F,G+F,X,YAN. Sin CSV/PNG externos ni cambios de originales.


In [ ]:
from pathlib import Path
import sys,json,hashlib
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace
from IPython.display import display,Markdown
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/wine_fermentation').is_dir())
assert Path(sys.prefix).name.lower()=='fermentation'
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
from wine_fermentation.models.historical import co2 as gas,kinetics
from wine_fermentation.models.historical.calibration import load,release_forward,Search
from wine_fermentation.comparison import observations
NAME='14_single_core_nutrition_dose'
protected=[p for folder in ('src','configs','data','notebooks','results') for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and not p.name.startswith(NAME+'.')]
before={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
old=__import__('nbformat').read(ROOT/'notebooks/diagnostics/09_historical_single_core.executed.ipynb',4)
stream=''.join(o.get('text','') for o in old.cells[4].outputs if o.output_type=='stream')
records=json.JSONDecoder().raw_decode(stream.split('PARAMETROS_COMPLETOS_SIN_EXPORTACION\n')[1])[0]
p0=next(r['parametros'] for r in records if r['caso']=='Propuesta · un core, sin O₂' and r['etapa']=='E1')
assert p0['A']==.3
theta=kinetics.normalize_parameters(json.loads((ROOT/'configs/theta/theta_nv2.json').read_text()))
theta_before=theta.copy();assert 0<theta['Yeg']<=.5114 and 0<theta['Yef']<=.5114
v=load(ROOT);LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018']
batches=dict(v.batches);b=batches['LAB017'];events=b.pulses['N'];assert np.isclose(sum(d for t,d in events),.28)
batches['LAB017']=replace(b,pulses={**b.pulses,'N':((min(t for t,d in events),.28),)})
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','co2_raw_g_l_h'])
obs=observations(raw);support={lab:obs[lab].loc[obs[lab].time_h.between(0,v.horizons[lab])].copy() for lab in LABS}
contexts={};calls=0
for lab,b in batches.items():
 f=gas.integrate_core(b,theta,b.time);calls+=1
 consumed=(f.G+f.F).iloc[0]-(f.G+f.F).to_numpy();z=np.clip((consumed-2.)/8.,0,1)
 contexts[lab]=dict(time=f.index.to_numpy(),dt=np.diff(f.index),gate=z*z*(3-2*z),q=gas.production(theta,b,f),
  sat_base=gas.csat(b,f,{'CO2sat_scale':1.}),events=b.pulses['N'],core=f)
assert calls==6 and all('q0' not in x and 'core0' not in x for x in contexts.values())
display(pd.Series(theta,name='theta_nv2 congelado'))
display(pd.Series(p0,name='Referencia E1; ahora A se fija en cero'))
windows={}
for lab in LABS:
 pulse=batches[lab].pulses['N'][0][0];end=v.horizons[lab]
 windows[lab]={'Registro':(0,end),'Arranque':(0,pulse),'Pico posterior':(pulse,min(pulse+24,end)),'Cola tardía':(pulse+32,end)}
display(pd.DataFrame([dict(LAB=lab,eventos=batches[lab].pulses['N'],corte=v.horizons[lab],**batches[lab].initials) for lab in LABS]))
display(pd.DataFrame([dict(LAB=lab,tramo=w,inicio=ab[0],fin=ab[1]) for lab,ww in windows.items() for w,ab in ww.items()]))

from wine_fermentation.data import natural_batches
_,training,_=natural_batches(ROOT)
display(pd.DataFrame([dict(LAB=b.batch,eventos_N=b.pulses['N']) for b in training]))


In [ ]:
from wine_fermentation.comparison import chemical_observations,CHEMICAL_COLUMNS
base={**p0,'kCO2_release_h':p0['kCO2_release_h']/4,'A':.3}
assert base['tau_rise_h']==4 and base['tau_decay_h']==12
fractions=[1.,.75,.5];curves={};chemistry={};rows=[];dose_table=[];balance=[]
# Existing context from setup provides baseline core, reused only at100%.
for fraction in fractions:
 for lab in LABS:
  original_batch=batches[lab]
  b=replace(original_batch,pulses={**original_batch.pulses,'N':tuple((t,d*fraction) for t,d in original_batch.pulses['N'])})
  assert b.initials==original_batch.initials
  for t,d in b.pulses['N']:dose_table.append(dict(fraccion=fraction,LAB=lab,hora=t,YAN_mg_L=d*1000))
  if fraction==1.:f=contexts[lab]['core'].copy()
  else:f=gas.integrate_core(b,theta,b.time);calls+=1
  f['GF']=f.G+f.F;chemistry[fraction,lab]=f
  t=f.index.to_numpy();dt=np.diff(t);consumed=f.GF.iloc[0]-f.GF.to_numpy()
  z=np.clip((consumed-2.)/8.,0,1);gate=z*z*(3-2*z)
  O=np.zeros(len(t));ini=b.initials
  O[0]=base['O2_initial_scale']*gas.o2_saturation_mg_l(b.temperature_c[0],ini['E'],ini['G'],ini['F'])
  for i,step in enumerate(dt):
   uptake=base['O2_qmax_mg_gdw_h']*f.X.iloc[i]*O[i]/(.25+O[i]);O[i+1]=max(O[i]-step*uptake,0.)
  factor=.08+.92*.75**2/(.75**2+O**2)
  r,M=Search.histories(t,b.pulses['N'],base);a=1+(base['pulse_activity_gain']-1)*r
  src=gate*a*factor*M*gas.production(theta,b,f)
  sat=gas.csat(b,f,{'CO2sat_scale':1.})
  pred,cd,qgas=release_forward(dict(source=src,dt=dt,sat_base=sat),base)
  err=np.max(abs(np.diff(cd)-dt*(src[:-1]-qgas[1:])));assert err<1e-9;balance.append(err)
  curves[fraction,lab]=pd.DataFrame(dict(qpred=pred,Cd=cd,qprod=src,M=M),index=t)
  o=support[lab]
  for window in ['Registro','Pico posterior','Cola tardía']:
   start,end=windows[lab][window];pts=o.loc[o.time_h.between(start,end)]
   tt=pts.time_h.to_numpy();y=pts.co2_raw_g_l_h.to_numpy();yp=np.interp(tt,t,pred);e=yp-y
   rows.append(dict(fraccion=fraction,LAB=lab,rol='Calibración previa' if lab in LABS[:3] else 'Holdout previo',
    tramo=window,RMSE=np.sqrt(np.mean(e*e)),sesgo=e.mean(),pico_predicho=yp.max(),pico_medido=y.max(),
    hora_pico_predicho=tt[np.argmax(yp)],hora_pico_medida=tt[np.argmax(y)]))
metrics=pd.DataFrame(rows)
display(pd.Series(base,name='Parámetros congelados'))
display(pd.DataFrame(dose_table));display(metrics.round(6))
display(metrics.groupby(['tramo','rol','fraccion']).RMSE.mean().unstack('fraccion').round(6))
assert calls==18 and theta==theta_before
ref=metrics.loc[metrics.fraccion.eq(1)&metrics.tramo.eq('Registro')].groupby('rol').RMSE.mean()
assert np.isclose(ref['Holdout previo'],.1373986258,atol=1e-8)
assert max(balance)<1e-9


In [ ]:
def finish(fig,axes,title,subtitle):
 items={}
 for ax in axes.flat:
  hh,ll=ax.get_legend_handles_labels()
  for h,l in zip(hh,ll):items.setdefault(l,h)
 fig.suptitle(title+'\n'+subtitle,y=.99,fontsize=14)
 fig.legend(items.values(),items.keys(),loc='upper center',bbox_to_anchor=(.5,.89),ncol=3,title='Dosis y mediciones · líneas verticales: nutrición')
 fig.subplots_adjust(top=.75,hspace=.4,wspace=.25);plt.show()
for fraction in [.75,.5]:
 for window in ['Registro','Pico posterior','Cola tardía']:
  fig,axes=plt.subplots(2,3,figsize=(16,9))
  for ax,lab in zip(axes.flat,LABS):
   start,end=windows[lab][window];o=support[lab];o=o.loc[o.time_h.between(start,end)]
   ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.1,label='Medianas horarias del sensor')
   for ff,color in [(1.,'#0072B2'),(fraction,'#009E73')]:
    f=curves[ff,lab];m=(f.index>=start)&(f.index<=end)
    ax.plot(f.index[m],f.qpred[m],color=color,label=f'Pulso al {100*ff:g}%')
   for tp,d in batches[lab].pulses['N']:
    if start<=tp<=end:ax.axvline(tp,color='.5',ls=':',lw=.8)
   ax.set(title=lab+(' · Calibración previa CO₂' if lab in LABS[:3] else ' · Holdout previo CO₂'),
    xlabel='Tiempo desde inoculación [h]',ylabel='CO₂ [g/(L·h)]',xlim=(start,end));ax.set_ylim(bottom=0);ax.grid(alpha=.25)
  finish(fig,axes,f'Pulso al {fraction*100:g}% · CO₂ · '+window,'theta_nv2 y todos los coeficientes fijos')
 for key,title,scale,unit in [('E','Etanol',1,'g/L'),('G','Glucosa',1,'g/L'),('F','Fructosa',1,'g/L'),('GF','Glucosa + fructosa',1,'g/L'),('X','Biomasa viable',1,'g/L'),('N','YAN',1000,'mg/L')]:
  fig,axes=plt.subplots(2,3,figsize=(16,9))
  for ax,lab in zip(axes.flat,LABS):
   for ff,color in [(1.,'#0072B2'),(fraction,'#009E73')]:
    f=chemistry[ff,lab];ax.plot(f.index,f[key]*scale,color=color,label=f'Pulso al {100*ff:g}%')
   if lab in LABS[:3]:
    pts=chemical_observations(v.chem,lab,key);tt=pts.t_h.to_numpy();yy=pts[CHEMICAL_COLUMNS[key]].to_numpy();labelf='Muestras experimentales'
   elif key=='GF':
    pts=v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l']);tt=pts.t_h.to_numpy();yy=2.5*(pts.density_g_l.to_numpy()-1000);labelf='G+F por densidad'
   elif key=='X':
    pts=v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h','X_viable_g_l']);tt=pts.t_h.to_numpy();yy=pts.X_viable_g_l.to_numpy();labelf='Biomasa experimental'
   else:tt=np.array([]);yy=np.array([]);labelf='Sin muestras disponibles'
   if len(tt):ax.scatter(tt,yy*scale,color='black',s=24,zorder=5,label=labelf)
   else:ax.text(.98,.94,labelf,ha='right',va='top',transform=ax.transAxes,color='.4')
   for tp,d in batches[lab].pulses['N']:ax.axvline(tp,color='.5',ls=':',lw=.8)
   ax.set(title=lab+(' · Validación química' if lab in LABS[:3] else ' · Validación offline'),
    xlabel='Tiempo desde inoculación [h]',ylabel=title+' ['+unit+']',xlim=(0,max(v.horizons[lab],max(tt) if len(tt) else 0)));ax.set_ylim(bottom=0);ax.grid(alpha=.25)
  finish(fig,axes,f'Pulso al {fraction*100:g}% · '+title,'Se modifica el aporte de N, no los parámetros cinéticos')


In [ ]:
assert theta==theta_before and calls==18
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in before.items())
print('Originales intactos; sólo pulsos reducidos en memoria, química y M coherentes; ninguna recalibración.')
print('SUMMARY_JSON')
print(metrics.groupby(['tramo','rol','fraccion']).RMSE.mean().reset_index().to_json(orient='records',force_ascii=False))


## Lectura del ensayo y dosis

Pulsos de YAN realmente usados por los lectores actuales:80mg/L en cada LAB004–009 (comprobados con natural_batches);80mg/L en LAB019–021;140/280/140mg/L en LAB016/017/018. Los últimos son supuestos de implementación, no mediciones directas.

Las mismas masas nominales1gSFX+0.4gFDA en2L aparecen en protocolo004–009 y en Nutricion de019–021. Su concentración nominal es0.5g/L SFX+0.2g/L FDA por pulso. La implementación histórica del gas asignó140mg/L usando20% para ambos productos, mientras el workbook asigna80mg/L. Es una discrepancia de conversión y procedencia; no demuestra que se administrara mayor masa en016/018. Las masas reales de016–018 siguen pendientes de confirmación, incluyendo la alternativa histórica0.8gSFX+0.4gFDA; LAB017 está confirmado como un solo pulso de dosis doble.

Escenarios al75%: LAB019–02160mg/L; LAB016/018105mg/L; LAB017210mg/L. Escenarios al50%:40,70 y140mg/L respectivamente. No se cambia el nutriente inicial. Si ambos productos de la receta nominal1g+0.4g se redujeran proporcionalmente,75% correspondería a0.75g+0.30g y50% a0.50g+0.20g; no son reconstrucciones de masas efectivamente aplicadas.

RMSE ordinario medio de CO₂, g/(L·h):

| Fracción de dosis | Registro LAB019–021 | Registro LAB016–018 | Pico LAB016–018 | Cola LAB016–018 |
|---|---:|---:|---:|---:|
| 100% | 0.128003 | 0.137399 | 0.227531 | 0.069878 |
| 75% | 0.127174 | 0.115330 | 0.147250 | 0.066131 |
| 50% | 0.129504 | 0.099856 | 0.066410 | 0.061646 |

En LAB016–018 la reducción disminuye el exceso del pico y sostiene algo más la cola. A diferencia de reducir A solamente, aquí también cambia la química: menor aporte de N reduce crecimiento posterior y puede prolongar consumo de azúcares. Eso explica que el efecto sobre cola no sea el mismo que al variar sólo M.

No hay mejora uniforme en calibración previa: al50% el error medio del registro LAB019–021 empeora ligeramente. En LAB021, con pico ya subestimado, bajar el aporte puede acentuar el déficit. Las figuras incluyen todas las muestras químicas disponibles, biomasa y G+F por densidad para valorar esa contrapartida.

El pulso reducido es una sensibilidad hipotética; una curva de CO₂ más próxima no demuestra que el registro experimental de dosis sea incorrecto. Primero se deben reconciliar masas de producto, aporte YAN de cada uno y volumen al dosificar. No se corrigen datos ni se recalibra theta_nv2 con estos escenarios. Se preservan todos los originales y no se exportan resultados CSV/PNG al proyecto.
